# Reciter TTS — упаковка Vosk-TTS (русский многоголосый VITS) для Android

Скачивает **vosk-model-tts-ru-0.7-multi** (alphacep): 5 русских голосов
(3 жен. + 2 муж.), словарные ударения, VITS 22.05 кГц, RTF < 1 на телефоне.
Даёт прослушать все голоса и пакует ZIP с `model.json`
(`architecture: "vosk-vits"`) для импорта в приложении.

Важно: берём именно **0.7-multi** — модели 0.8/0.9 (multistream) требуют
BERT-контекстер, который на телефон не переносим. GPU не нужен.


In [ ]:
%pip install -q vosk-tts onnxruntime soundfile

from vosk_tts import Model, Synth

MODEL_NAME = "vosk-model-tts-ru-0.7-multi"
model = Model(model_name=MODEL_NAME)   # скачает в ~/.cache/vosk
synth = Synth(model)

import os, glob
ROOT = glob.glob(os.path.expanduser(f"~/.cache/vosk/{MODEL_NAME}"))[0]
print("модель:", ROOT)
print("файлы:", sorted(os.listdir(ROOT)))
assert not os.path.isdir(os.path.join(ROOT, "bert")), \
    "модель со встроенным BERT: для Android нужна 0.7-multi (без каталога bert)"



## Прослушать все 5 голосов

Та же фраза каждым диктором. Выбери, кого оставить / кто будет по умолчанию.


In [ ]:
from IPython.display import Audio, display

TEXT = "Привет! Это проверка русского синтеза. Съешь ещё этих мягких французских булок, да выпей чаю."

VOICES = [
    (0, "Женский 1"), (1, "Женский 2"), (2, "Женский 3"),
    (3, "Мужской 1"), (4, "Мужской 2"),
]
os.makedirs("/content/vosk_probe", exist_ok=True)
for sid, label in VOICES:
    out = f"/content/vosk_probe/sid{sid}.wav"
    synth.synth(TEXT, out, speaker_id=sid)
    print(f"sid={sid}  {label}")
    display(Audio(out))


## Упаковка ZIP для приложения

Кладём `model.onnx`, `dictionary`, `config.json` + манифест `model.json`.
В приложении: Модели → «Выбрать локальный ZIP».


In [ ]:
import json, zipfile

voices = []
for sid, label in VOICES:
    voices.append({
        "id": f"vosk_ru_{sid}", "locale": "ru-RU",
        "displayName": f"{label} (Vosk)", "speakerId": sid,
    })

def mb(p): return round(os.path.getsize(p) / 1024**2, 2)

manifest = {
    "schemaVersion": 1, "id": "vosk-tts-ru-multi",
    "displayName": "Vosk TTS Russian (5 голосов)",
    "family": "vosk", "architecture": "vosk-vits",
    "sampleRateHz": 22050, "runtime": "onnxruntime",
    "tokenizer": {"type": "vosk-dictionary", "files": ["dictionary", "config.json"]},
    "files": [
        {"filename": "model.onnx", "role": "TALKER", "required": True,
         "sizeMb": mb(os.path.join(ROOT, "model.onnx"))},
        {"filename": "dictionary", "role": "VOCODER", "required": True,
         "sizeMb": mb(os.path.join(ROOT, "dictionary"))},
    ],
    "voices": voices,
}
with open(os.path.join(ROOT, "model.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

OUT = "/content/reciter-vosk-ru-android.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ["model.onnx", "dictionary", "config.json", "model.json"]:
        zf.write(os.path.join(ROOT, name), name)
print(f"архив: {os.path.getsize(OUT)/1024**2:.1f} MB -> {OUT}")

try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass
